# 14 — Budgeted historical odds expansion v2

This notebook is the next step after `13_data_source_audit`.

Goal:
- spend remaining The Odds API credits in a controlled way;
- expand historical h2h odds for competitions where the source appears to work;
- add more time snapshots for timing and closing-line-value analysis;
- save an expanded `data/processed/historical_odds_aggregated.csv`.

Important notes:
- this notebook does not use API-Football or Sportmonks;
- it only uses The Odds API historical endpoints;
- it has a hard budget cap;
- already downloaded `event_id + snapshot_label` pairs are skipped.

After running this notebook, re-run:
1. `03_join_market_train_and_backtest.ipynb`
2. `07_walkforward_model_vs_market_backtest.ipynb`
3. `08_edge_selector_grid_search.ipynb`
4. `09_robust_edge_validation.ipynb`
5. `11_market_anchored_correction_and_edge_selector.ipynb`
6. `12_current_wc2026_paper_picks.ipynb`


In [ ]:
# Cell 1. Paste The Odds API key here.

ODDS_API_KEY = "PASTE_THE_ODDS_API_KEY_HERE"


In [ ]:
# Cell 2. Imports and helpers.

from pathlib import Path
import json
import re
import time
import zipfile

import requests
import numpy as np
import pandas as pd

DATA_DIR = Path("data")
PROCESSED_DIR = DATA_DIR / "processed"
OUT_DIR = PROCESSED_DIR / "14_budgeted_historical_odds_expansion_v2"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

RUN_TIMESTAMP_UTC = pd.Timestamp.utcnow().isoformat()

ODDS_BASE = "https://api.the-odds-api.com/v4"

TEAM_REPLACEMENTS = {
    "USA": "United States",
    "US": "United States",
    "Korea Republic": "South Korea",
    "Türkiye": "Turkey",
    "Czechia": "Czech Republic",
    "Bosnia and Herzegovina": "Bosnia & Herzegovina",
    "Ivory Coast": "Côte d'Ivoire",
    "Cote d Ivoire": "Côte d'Ivoire",
}

def has_key(value):
    if value is None:
        return False
    value = str(value).strip()
    return value != "" and not value.startswith("PASTE_")

def mask_key(value):
    if not has_key(value):
        return "missing"
    value = str(value)
    return f"present_len_{len(value)}_last4_{value[-4:]}"

def norm_team(name):
    if pd.isna(name):
        return ""
    name = str(name).strip()
    name = re.sub(r"\s+", " ", name)
    return TEAM_REPLACEMENTS.get(name, name)

def save_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2, ensure_ascii=False, default=str)
    return path

def safe_get_json(url, params=None, timeout=60):
    try:
        response = requests.get(url, params=params or {}, timeout=timeout)
        try:
            data = response.json()
        except Exception:
            data = {"raw_text": response.text[:3000]}
        return response.status_code, data, dict(response.headers)
    except Exception as exc:
        return -1, {"error": str(exc)}, {}

def sanitize_params(params):
    out = dict(params or {})
    for key in list(out.keys()):
        if str(key).lower() in {"apikey", "api_key", "key", "token"}:
            out[key] = mask_key(out[key])
    return out

def usage_headers(headers):
    return {
        "x_requests_remaining": headers.get("x-requests-remaining"),
        "x_requests_used": headers.get("x-requests-used"),
        "x_requests_last": headers.get("x-requests-last"),
    }

report_rows = []

def add_report_row(check, endpoint, params, status, data, headers, path):
    row_count = 0
    if isinstance(data, list):
        row_count = len(data)
    elif isinstance(data, dict):
        value = data.get("data")
        if isinstance(value, list):
            row_count = len(value)

    error = ""
    if isinstance(data, dict):
        error = str(data.get("message", data.get("error", "")))[:500]

    report_rows.append({
        "run_timestamp_utc": RUN_TIMESTAMP_UTC,
        "check": check,
        "endpoint": endpoint,
        "params_masked": json.dumps(sanitize_params(params)),
        "status_code": status,
        "rows": row_count,
        "error": error,
        "cache_path": str(path),
        **usage_headers(headers),
    })

def odds_get(path, params=None, name="request"):
    params = dict(params or {})
    params["apiKey"] = ODDS_API_KEY

    url = f"{ODDS_BASE}/{path.lstrip('/')}"

    status, data, headers = safe_get_json(url, params=params)

    safe_name = re.sub(r"[^a-zA-Z0-9_]+", "_", name)
    out_path = OUT_DIR / "raw" / f"{safe_name}.json"

    save_json(out_path, {
        "url_path": path,
        "params_masked": sanitize_params(params),
        "status_code": status,
        "headers": headers,
        "data": data,
    })

    add_report_row(
        check=name,
        endpoint=path,
        params=params,
        status=status,
        data=data,
        headers=headers,
        path=out_path,
    )

    return status, data, headers

if not has_key(ODDS_API_KEY):
    raise ValueError("Paste ODDS_API_KEY in Cell 1 first.")

print("Setup OK:", ODDS_BASE)


In [ ]:
# Cell 3. Load existing historical odds and audit context.

existing_paths = [
    PROCESSED_DIR / "historical_odds_aggregated.csv",
    PROCESSED_DIR / "historical_odds" / "historical_odds_aggregated.csv",
    PROCESSED_DIR / "06_multi_comp_historical_odds" / "combined_historical_odds_aggregated.csv",
]

existing_parts = []

for path in existing_paths:
    if path.exists():
        try:
            df = pd.read_csv(path)
            df["source_existing_file"] = str(path)
            existing_parts.append(df)
            print("Loaded existing:", path, df.shape)
        except Exception as exc:
            print("Failed:", path, exc)

if existing_parts:
    existing_odds = pd.concat(existing_parts, ignore_index=True, sort=False)
    existing_odds = existing_odds.drop_duplicates(
        ["event_id", "snapshot_label"],
        keep="first",
    )
else:
    existing_odds = pd.DataFrame()

already_have = set()

if len(existing_odds) > 0:
    for _, row in existing_odds.iterrows():
        already_have.add((
            str(row.get("event_id")),
            str(row.get("snapshot_label")),
        ))

print("existing aggregated snapshots:", len(existing_odds))
print("existing event+snapshot keys:", len(already_have))

display(existing_odds.head())


In [ ]:
# Cell 4. Expansion config and budget.

# Your audit showed around 11.5k remaining credits.
# We keep this conservative so current WC2026 monitoring still has budget.
MAX_CREDITS_THIS_RUN = 4500

REGION = "eu"
MARKETS = "h2h"

# More snapshots = better timing/CLV analysis.
# Existing T-24/T-3/T-1 will be skipped if already present.
SNAPSHOT_OFFSETS_HOURS = [
    168,  # T-7d
    48,   # T-48h
    24,   # T-24h
    12,   # T-12h
    6,    # T-6h
    3,    # T-3h
    1,    # T-1h
]

# Based on audit:
# - current World Cup works
# - UEFA European Championship key exists
# - Africa Cup, Gold Cup, Nations League keys exist
# - soccer_uefa_euro / copa_america / asian_cup returned unknown in probe,
#   so they are not priority here.
COMPETITION_CONFIGS = [
    {
        "label": "world_cup_2022",
        "sport_key": "soccer_fifa_world_cup",
        "start": "2022-11-20 12:00:00",
        "end": "2022-12-18 12:00:00",
        "freq": "2D",
        "max_events": 64,
        "priority": 1,
    },
    {
        "label": "euro_2024",
        "sport_key": "soccer_uefa_european_championship",
        "start": "2024-06-14 12:00:00",
        "end": "2024-07-14 12:00:00",
        "freq": "2D",
        "max_events": 51,
        "priority": 1,
    },
    {
        "label": "euro_2021",
        "sport_key": "soccer_uefa_european_championship",
        "start": "2021-06-11 12:00:00",
        "end": "2021-07-11 12:00:00",
        "freq": "2D",
        "max_events": 51,
        "priority": 2,
    },
    {
        "label": "afcon_2023",
        "sport_key": "soccer_africa_cup_of_nations",
        "start": "2024-01-13 12:00:00",
        "end": "2024-02-11 12:00:00",
        "freq": "3D",
        "max_events": 52,
        "priority": 2,
    },
    {
        "label": "afcon_2021",
        "sport_key": "soccer_africa_cup_of_nations",
        "start": "2022-01-09 12:00:00",
        "end": "2022-02-06 12:00:00",
        "freq": "3D",
        "max_events": 52,
        "priority": 3,
    },
    {
        "label": "gold_cup_2023",
        "sport_key": "soccer_concacaf_gold_cup",
        "start": "2023-06-24 12:00:00",
        "end": "2023-07-16 12:00:00",
        "freq": "3D",
        "max_events": 31,
        "priority": 3,
    },
    {
        "label": "gold_cup_2021",
        "sport_key": "soccer_concacaf_gold_cup",
        "start": "2021-07-10 12:00:00",
        "end": "2021-08-01 12:00:00",
        "freq": "3D",
        "max_events": 31,
        "priority": 4,
    },
    {
        "label": "nations_league_sample_2022_2024",
        "sport_key": "soccer_uefa_nations_league",
        "start": "2022-06-01 12:00:00",
        "end": "2024-11-20 12:00:00",
        "freq": "30D",
        "max_events": 80,
        "priority": 4,
    },
]

# Discovery calls are cheap.
discovery_calls = 0

for cfg in COMPETITION_CONFIGS:
    dates = pd.date_range(
        cfg["start"],
        cfg["end"],
        freq=cfg["freq"],
        tz="UTC",
    )
    discovery_calls += len(dates)

# Historical odds snapshot is about 10 credits for 1 region x 1 market.
MAX_ODDS_SNAPSHOT_CALLS = max(
    0,
    int((MAX_CREDITS_THIS_RUN - discovery_calls - 10) // 10),
)

budget_report = pd.DataFrame([{
    "max_credits_this_run": MAX_CREDITS_THIS_RUN,
    "discovery_calls_estimate": discovery_calls,
    "max_odds_snapshot_calls": MAX_ODDS_SNAPSHOT_CALLS,
    "estimated_max_credits": discovery_calls + 10 * MAX_ODDS_SNAPSHOT_CALLS,
    "region": REGION,
    "markets": MARKETS,
    "snapshots": str(SNAPSHOT_OFFSETS_HOURS),
}])

budget_report.to_csv(OUT_DIR / "budget_report.csv", index=False)

display(budget_report)

if MAX_ODDS_SNAPSHOT_CALLS <= 0:
    raise ValueError("Budget too small for odds snapshots.")


In [ ]:
# Cell 5. Discover historical events.

event_rows = []
probe_rows = []

for cfg in sorted(COMPETITION_CONFIGS, key=lambda x: x["priority"]):
    dates = pd.date_range(
        cfg["start"],
        cfg["end"],
        freq=cfg["freq"],
        tz="UTC",
    )

    before = len(event_rows)

    for ts in dates:
        ts_iso = ts.isoformat().replace("+00:00", "Z")

        status, data, headers = odds_get(
            f"historical/sports/{cfg['sport_key']}/events",
            params={
                "dateFormat": "iso",
                "date": ts_iso,
            },
            name=f"events_{cfg['label']}_{ts_iso}",
        )

        events = []
        if isinstance(data, dict):
            events = data.get("data", []) or []

        probe_rows.append({
            "label": cfg["label"],
            "sport_key": cfg["sport_key"],
            "timestamp": ts_iso,
            "status": status,
            "rows": len(events),
            "error": (
                str(data.get("message", data.get("error", "")))[:300]
                if isinstance(data, dict)
                else ""
            ),
            **usage_headers(headers),
        })

        for event in events:
            event_rows.append({
                "label": cfg["label"],
                "sport_key": cfg["sport_key"],
                "priority": cfg["priority"],
                "event_id": str(event.get("id")),
                "commence_time": event.get("commence_time"),
                "home_team": norm_team(event.get("home_team")),
                "away_team": norm_team(event.get("away_team")),
                "discovery_timestamp": ts_iso,
            })

        time.sleep(0.12)

    print(
        cfg["label"],
        cfg["sport_key"],
        "new event rows:",
        len(event_rows) - before,
    )

event_probe = pd.DataFrame(probe_rows)
event_probe.to_csv(OUT_DIR / "event_discovery_probe.csv", index=False)

events_df = pd.DataFrame(event_rows)

if len(events_df) > 0:
    events_df = events_df.drop_duplicates(["sport_key", "event_id"])
    events_df["commence_time"] = pd.to_datetime(
        events_df["commence_time"],
        utc=True,
        errors="coerce",
    )
    events_df = events_df.sort_values(
        ["priority", "sport_key", "commence_time"],
    )

    limited = []
    for cfg in COMPETITION_CONFIGS:
        part = events_df[
            (events_df["label"] == cfg["label"])
            & (events_df["sport_key"] == cfg["sport_key"])
        ].head(cfg["max_events"])
        limited.append(part)

    events_df = pd.concat(limited, ignore_index=True)
    events_df = events_df.drop_duplicates(["sport_key", "event_id"])
    events_df = events_df.sort_values(
        ["priority", "sport_key", "commence_time"],
    )

events_df.to_csv(OUT_DIR / "events_discovered_limited.csv", index=False)

display(event_probe.groupby(["label", "sport_key"]).agg(
    calls=("timestamp", "count"),
    http_200=("status", lambda s: int((s == 200).sum())),
    rows=("rows", "sum"),
).reset_index())

display(events_df.head(50))
print("limited events:", len(events_df))


In [ ]:
# Cell 6. Build missing snapshot plan.

snapshot_rows = []

for _, event in events_df.iterrows():
    kickoff = event["commence_time"]
    if pd.isna(kickoff):
        continue

    for offset in SNAPSHOT_OFFSETS_HOURS:
        snapshot_label = f"T_minus_{offset}h"
        pair = (str(event["event_id"]), snapshot_label)

        if pair in already_have:
            continue

        snapshot_ts = kickoff - pd.Timedelta(hours=offset)
        snapshot_iso = snapshot_ts.isoformat().replace("+00:00", "Z")

        snapshot_rows.append({
            "label": event["label"],
            "sport_key": event["sport_key"],
            "priority": event["priority"],
            "event_id": str(event["event_id"]),
            "commence_time": kickoff,
            "home_team": event["home_team"],
            "away_team": event["away_team"],
            "snapshot_label": snapshot_label,
            "snapshot_offset_hours": offset,
            "snapshot_timestamp": snapshot_iso,
        })

snapshot_plan = pd.DataFrame(snapshot_rows)

if len(snapshot_plan) > 0:
    # Prioritize:
    # 1. competitions with higher priority
    # 2. T-24 / T-6 / T-1 first
    offset_priority = {24: 1, 6: 2, 1: 3, 3: 4, 12: 5, 48: 6, 168: 7}
    snapshot_plan["offset_priority"] = snapshot_plan[
        "snapshot_offset_hours"
    ].map(offset_priority).fillna(99)

    snapshot_plan = snapshot_plan.sort_values(
        ["priority", "offset_priority", "commence_time"],
    ).reset_index(drop=True)

snapshot_plan_limited = snapshot_plan.head(MAX_ODDS_SNAPSHOT_CALLS).copy()

snapshot_plan.to_csv(OUT_DIR / "snapshot_plan_all_missing.csv", index=False)
snapshot_plan_limited.to_csv(
    OUT_DIR / "snapshot_plan_limited_to_fetch.csv",
    index=False,
)

print("all missing snapshots:", len(snapshot_plan))
print("limited snapshots to fetch:", len(snapshot_plan_limited))
display(snapshot_plan_limited.head(50))


In [ ]:
# Cell 7. Fetch selected historical odds snapshots.

def parse_event_odds(plan_row, data):
    rows = []

    if not isinstance(data, dict):
        return rows

    event_obj = data.get("data", data)
    event_list = event_obj if isinstance(event_obj, list) else [event_obj]

    for event in event_list:
        home_team = norm_team(event.get("home_team", plan_row["home_team"]))
        away_team = norm_team(event.get("away_team", plan_row["away_team"]))
        commence_time = event.get("commence_time", plan_row["commence_time"])

        for bookmaker in event.get("bookmakers", []) or []:
            bookmaker_key = bookmaker.get("key")
            bookmaker_title = bookmaker.get("title")

            for market in bookmaker.get("markets", []) or []:
                if market.get("key") != "h2h":
                    continue

                prices = {}

                for outcome in market.get("outcomes", []) or []:
                    name = norm_team(outcome.get("name", ""))
                    price = pd.to_numeric(
                        outcome.get("price"),
                        errors="coerce",
                    )

                    if pd.isna(price):
                        continue

                    if name == home_team:
                        prices["home_odds"] = float(price)
                    elif name == away_team:
                        prices["away_odds"] = float(price)
                    elif str(name).lower() in {"draw", "tie"}:
                        prices["draw_odds"] = float(price)

                if {"home_odds", "draw_odds", "away_odds"} <= set(prices):
                    rows.append({
                        "label": plan_row["label"],
                        "sport_key": plan_row["sport_key"],
                        "event_id": str(plan_row["event_id"]),
                        "snapshot_label": plan_row["snapshot_label"],
                        "snapshot_timestamp": plan_row["snapshot_timestamp"],
                        "snapshot_offset_hours": plan_row[
                            "snapshot_offset_hours"
                        ],
                        "commence_time": commence_time,
                        "home_team": home_team,
                        "away_team": away_team,
                        "bookmaker_key": bookmaker_key,
                        "bookmaker_title": bookmaker_title,
                        **prices,
                    })

    return rows

raw_rows = []

for i, plan_row in snapshot_plan_limited.iterrows():
    status, data, headers = odds_get(
        (
            f"historical/sports/{plan_row['sport_key']}/events/"
            f"{plan_row['event_id']}/odds"
        ),
        params={
            "regions": REGION,
            "markets": MARKETS,
            "oddsFormat": "decimal",
            "dateFormat": "iso",
            "date": plan_row["snapshot_timestamp"],
        },
        name=(
            f"odds_{plan_row['label']}_{plan_row['event_id']}_"
            f"{plan_row['snapshot_label']}"
        ),
    )

    raw_rows.extend(parse_event_odds(plan_row, data))

    if (i + 1) % 25 == 0:
        print("Fetched snapshots:", i + 1)

    time.sleep(0.12)

new_raw = pd.DataFrame(raw_rows)

new_raw.to_csv(
    OUT_DIR / "new_raw_bookmaker_odds.csv",
    index=False,
)

print("new raw bookmaker rows:", len(new_raw))
display(new_raw.head(30))


In [ ]:
# Cell 8. Aggregate and combine.

if len(new_raw) == 0:
    new_agg = pd.DataFrame()
else:
    agg_rows = []

    group_cols = [
        "label",
        "sport_key",
        "event_id",
        "snapshot_label",
        "snapshot_timestamp",
        "snapshot_offset_hours",
        "commence_time",
        "home_team",
        "away_team",
    ]

    for keys, group in new_raw.groupby(group_cols):
        row = dict(zip(group_cols, keys))

        avg_home = group["home_odds"].mean()
        avg_draw = group["draw_odds"].mean()
        avg_away = group["away_odds"].mean()

        raw = np.array([
            1.0 / avg_away,
            1.0 / avg_draw,
            1.0 / avg_home,
        ])

        probs = raw / raw.sum()

        row.update({
            "n_bookmakers": group["bookmaker_key"].nunique(),
            "avg_home_odds": float(avg_home),
            "avg_draw_odds": float(avg_draw),
            "avg_away_odds": float(avg_away),
            "best_home_odds": float(group["home_odds"].max()),
            "best_draw_odds": float(group["draw_odds"].max()),
            "best_away_odds": float(group["away_odds"].max()),
            "market_margin_avg": float(raw.sum() - 1.0),
            "market_p_away_devig": float(probs[0]),
            "market_p_draw_devig": float(probs[1]),
            "market_p_home_devig": float(probs[2]),
        })

        agg_rows.append(row)

    new_agg = pd.DataFrame(agg_rows)

parts = []

if len(existing_odds) > 0:
    parts.append(existing_odds)

if len(new_agg) > 0:
    parts.append(new_agg)

if parts:
    combined = pd.concat(parts, ignore_index=True, sort=False)
    combined = combined.drop_duplicates(
        ["event_id", "snapshot_label"],
        keep="first",
    )
else:
    combined = pd.DataFrame()

new_agg.to_csv(
    OUT_DIR / "new_historical_odds_aggregated.csv",
    index=False,
)

combined.to_csv(
    OUT_DIR / "combined_historical_odds_aggregated.csv",
    index=False,
)

combined.to_csv(
    PROCESSED_DIR / "historical_odds_aggregated.csv",
    index=False,
)

print("new aggregated snapshots:", len(new_agg))
print("combined aggregated snapshots:", len(combined))
print("root file updated:", PROCESSED_DIR / "historical_odds_aggregated.csv")

display(combined.head())


In [ ]:
# Cell 9. Save report zip.

status_df = pd.DataFrame(report_rows)
status_df.to_csv(OUT_DIR / "the_odds_api_status.csv", index=False)

actual_credit_sum = None

if "x_requests_last" in status_df.columns:
    actual_credit_sum = pd.to_numeric(
        status_df["x_requests_last"],
        errors="coerce",
    ).fillna(0).sum()

summary = {
    "run_timestamp_utc": RUN_TIMESTAMP_UTC,
    "existing_aggregated_snapshots": int(len(existing_odds)),
    "events_discovered_limited": int(len(events_df)),
    "all_missing_snapshots": int(len(snapshot_plan)),
    "attempted_snapshot_calls": int(len(snapshot_plan_limited)),
    "new_raw_bookmaker_rows": int(len(new_raw)),
    "new_aggregated_snapshots": int(len(new_agg)),
    "combined_aggregated_snapshots": int(len(combined)),
    "max_credits_this_run": int(MAX_CREDITS_THIS_RUN),
    "actual_credit_sum_from_headers": (
        float(actual_credit_sum)
        if actual_credit_sum is not None
        else None
    ),
    "region": REGION,
    "markets": MARKETS,
    "snapshot_offsets_hours": SNAPSHOT_OFFSETS_HOURS,
}

save_json(OUT_DIR / "summary.json", summary)

zip_path = PROCESSED_DIR / "14_budgeted_historical_odds_expansion_v2_report_bundle.zip"

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for file in OUT_DIR.rglob("*"):
        if file.is_file():
            zf.write(file, arcname=file.relative_to(OUT_DIR))
    if (PROCESSED_DIR / "historical_odds_aggregated.csv").exists():
        zf.write(
            PROCESSED_DIR / "historical_odds_aggregated.csv",
            arcname="historical_odds_aggregated.csv",
        )

display(pd.DataFrame([summary]))
display(status_df.tail(20))

print("Created:", zip_path.resolve())
print("Next: rerun 03, 07, 08, 09, 11, 12.")
